# Context-Aware HiRISE Reconstruction Flow

This notebook walks through the full paired local/context masked-reconstruction pipeline on the HiRISE `.JP2` files in `data/`:

1. extract paired local and context crops,
2. inspect a `2048x2048` overview crop with local/context boxes overlaid,
3. train the context-aware reconstruction model for `5` epochs,
4. inspect reconstructed local patches,
5. stitch the reconstructed patches back together for image-level inspection.


In [ ]:
import csv
import math
import random
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import torch

if (Path.cwd() / 'model' / 'model.py').exists():
    VISION_BACKEND_DIR = Path.cwd().resolve()
    PROJECT_ROOT = VISION_BACKEND_DIR.parent
elif (Path.cwd() / 'AI4ExoMars' / 'vision_backend' / 'model' / 'model.py').exists():
    PROJECT_ROOT = (Path.cwd() / 'AI4ExoMars').resolve()
    VISION_BACKEND_DIR = PROJECT_ROOT / 'vision_backend'
else:
    raise RuntimeError('Run this notebook from the repo root or from AI4ExoMars/vision_backend.')

if str(VISION_BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(VISION_BACKEND_DIR))

from context_reconstruction import (
    collect_context_examples,
    create_context_patch_dataloaders,
    group_records_by_source,
    load_patch_records,
    reconstruct_dataset,
    stitch_patch_arrays,
    HiRISEContextPatchDataset,
)
from hirise_patchloader import open_image_reader, make_context_window_spec, pad_patch
from model.model import (
    ContextAwareConvNeXtSwinEncoder,
    ContextAwareMaskedReconstructionPretrainer,
)
from model.optimizers import create_optimizer, create_cosine_scheduler_with_warmup


## Configuration

In [ ]:
RUN_ARGS = {
    'input_dir': str(PROJECT_ROOT.parent / 'data'),
    'output_dir': str(PROJECT_ROOT.parent / 'data' / 'hirise_context_pairs'),
    'index_path': str(PROJECT_ROOT.parent / 'data' / 'hirise_context_pairs' / 'patch_index.csv'),
    'local_patch_size': 256,
    'context_window_size': 1024,
    'context_input_size': 256,
    'overview_size': 2048,
    'stride': 256,
    'pad_mode': 'edge',
    'pad_value': 0.0,
    'max_black_fraction': 0.25,
    'black_threshold': 0.0,
    'num_examples': 5,
    'epochs': 5,
    'batch_size': 8,
    'num_workers': 0,
    'learning_rate': 3e-4,
    'weight_decay': 1e-2,
    'warmup_fraction': 0.1,
    'local_base_channels': 48,
    'context_base_channels': 24,
    'context_dim': 256,
    'decoder_channels': 256,
    'mask_patch_size': 16,
    'mask_ratio': 0.6,
    'window_size': 8,
    'swin_depths': (2, 2, 2),
    'swin_num_heads': (4, 8, 16),
    'loss_type': 'l1',
    'val_fraction': 0.1,
    'test_fraction': 0.0,
    'use_stage32': True,
    'use_muon': False,
    'seed': 42,
}

RUN_ARGS


## Runtime Setup

In [ ]:
seed = RUN_ARGS['seed']
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

use_amp = device.type == 'cuda'
print('Using device:', device)


## 1. Extract Paired Local and Context Crops

This runs the patch loader with:
- `256x256` local patches,
- `1024x1024` raw context windows saved alongside them,
- black-border filtering,
- example previews for sanity checking.


In [ ]:
patchloader_cmd = [
    sys.executable,
    str(VISION_BACKEND_DIR / 'hirise_patchloader.py'),
    '--input-dir', RUN_ARGS['input_dir'],
    '--output-dir', RUN_ARGS['output_dir'],
    '--patch-size', str(RUN_ARGS['local_patch_size']),
    '--context-size', str(RUN_ARGS['context_window_size']),
    '--stride', str(RUN_ARGS['stride']),
    '--pad-mode', RUN_ARGS['pad_mode'],
    '--pad-value', str(RUN_ARGS['pad_value']),
    '--max-black-fraction', str(RUN_ARGS['max_black_fraction']),
    '--black-threshold', str(RUN_ARGS['black_threshold']),
    '--num-examples', str(RUN_ARGS['num_examples']),
    '--overwrite',
]

print('Running:', ' '.join(patchloader_cmd))
subprocess.run(patchloader_cmd, check=True)


In [ ]:
records = load_patch_records(RUN_ARGS['index_path'])
print(f'Kept paired patches: {len(records)}')
print('Unique source images:', len({record.source_image for record in records}))
pd.DataFrame([record.__dict__ for record in records]).head()


## 2. Inspect a `2048x2048` Overview Crop

The red box is the local `256x256` patch. The cyan box is the saved `1024x1024` context window centered around it.


In [ ]:
def normalize_for_display(array: np.ndarray) -> np.ndarray:
    arr = np.asarray(array, dtype=np.float32)
    if arr.ndim == 3 and arr.shape[2] == 1:
        arr = arr[:, :, 0]
    min_value = float(arr.min())
    max_value = float(arr.max())
    if max_value <= min_value:
        return np.zeros(arr.shape, dtype=np.float32)
    return (arr - min_value) / (max_value - min_value)

record = records[0]
center_y = record.top + record.patch_size // 2
center_x = record.left + record.patch_size // 2

overview_spec = make_context_window_spec(
    image_height=record.image_height,
    image_width=record.image_width,
    center_y=center_y,
    center_x=center_x,
    context_size=RUN_ARGS['overview_size'],
)

with open_image_reader(Path(record.source_image)) as reader:
    overview = reader.read_region(
        overview_spec.read_top,
        overview_spec.read_left,
        overview_spec.valid_height,
        overview_spec.valid_width,
    )

overview = pad_patch(
    overview,
    pad_top=overview_spec.pad_top,
    pad_bottom=overview_spec.pad_bottom,
    pad_left=overview_spec.pad_left,
    pad_right=overview_spec.pad_right,
    pad_mode=RUN_ARGS['pad_mode'],
    pad_value=RUN_ARGS['pad_value'],
)

overview_img = normalize_for_display(overview)
local_x = record.left - overview_spec.requested_left
local_y = record.top - overview_spec.requested_top
context_x = record.context_left - overview_spec.requested_left
context_y = record.context_top - overview_spec.requested_top

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(overview_img, cmap='gray')
ax.add_patch(
    patches.Rectangle(
        (context_x, context_y),
        record.context_size,
        record.context_size,
        linewidth=2.5,
        edgecolor='cyan',
        facecolor='none',
        label='1024 context window',
    )
)
ax.add_patch(
    patches.Rectangle(
        (local_x, local_y),
        record.patch_size,
        record.patch_size,
        linewidth=2.5,
        edgecolor='red',
        facecolor='none',
        label='256 local patch',
    )
)
ax.set_title(Path(record.source_image).name + ' overview (2048x2048)')
ax.axis('off')
ax.legend(loc='lower right')
plt.show()


## 3. Build DataLoaders

In [ ]:
loaders = create_context_patch_dataloaders(
    index_path=RUN_ARGS['index_path'],
    batch_size=RUN_ARGS['batch_size'],
    local_input_size=RUN_ARGS['local_patch_size'],
    context_input_size=RUN_ARGS['context_input_size'],
    val_fraction=RUN_ARGS['val_fraction'],
    test_fraction=RUN_ARGS['test_fraction'],
    num_workers=RUN_ARGS['num_workers'],
    seed=RUN_ARGS['seed'],
)

print('Train patches:', len(loaders.train_dataset))
print('Val patches:  ', len(loaders.val_dataset))
print('Test patches: ', 0 if loaders.test_dataset is None else len(loaders.test_dataset))


## 4. Build the Context-Aware Reconstruction Model

In [ ]:
use_stage32 = RUN_ARGS['use_stage32']
encoder = ContextAwareConvNeXtSwinEncoder(
    in_channels=1,
    local_base_channels=RUN_ARGS['local_base_channels'],
    context_base_channels=RUN_ARGS['context_base_channels'],
    context_dim=RUN_ARGS['context_dim'],
    use_stage32=use_stage32,
    swin_depths=tuple(RUN_ARGS['swin_depths']),
    swin_num_heads=tuple(RUN_ARGS['swin_num_heads']),
    window_size=RUN_ARGS['window_size'],
)

bottleneck_channels = RUN_ARGS['local_base_channels'] * (16 if use_stage32 else 8)
skip8_channels = RUN_ARGS['local_base_channels'] * 4

model = ContextAwareMaskedReconstructionPretrainer(
    encoder=encoder,
    in_channels=1,
    bottleneck_channels=bottleneck_channels,
    decoder_channels=RUN_ARGS['decoder_channels'],
    patch_size=RUN_ARGS['mask_patch_size'],
    mask_ratio=RUN_ARGS['mask_ratio'],
    bottleneck_index=-1,
    skip8_index=3,
    skip8_channels=skip8_channels,
    use_skip8=True,
    loss_type=RUN_ARGS['loss_type'],
).to(device)

optimizer = create_optimizer(
    model,
    lr=RUN_ARGS['learning_rate'],
    weight_decay=RUN_ARGS['weight_decay'],
    use_muon=RUN_ARGS['use_muon'],
)

total_steps = RUN_ARGS['epochs'] * max(len(loaders.train), 1)
warmup_steps = int(RUN_ARGS['warmup_fraction'] * total_steps)
scheduler = create_cosine_scheduler_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print('Total training steps:', total_steps)
print('Warmup steps:', warmup_steps)


## 5. Train for 5 Epochs

In [ ]:
from context_reconstruction import run_context_epoch

history_rows = []
best_val_loss = float('inf')
best_state = None

for epoch in range(1, RUN_ARGS['epochs'] + 1):
    train_loss = run_context_epoch(
        model=model,
        dataloader=loaders.train,
        device=device,
        optimizer=optimizer,
        scheduler=scheduler,
        use_amp=use_amp,
    )
    val_loss = run_context_epoch(
        model=model,
        dataloader=loaders.val,
        device=device,
        optimizer=None,
        use_amp=False,
    )
    current_lr = optimizer.param_groups[0]['lr']
    history_rows.append(
        {
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'lr': current_lr,
        }
    )
    print(
        f"[epoch {epoch:02d}/{RUN_ARGS['epochs']:02d}] "
        f"train_loss={train_loss:.6f} val_loss={val_loss:.6f} lr={current_lr:.3e}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {
            key: value.detach().cpu().clone() if isinstance(value, torch.Tensor) else value
            for key, value in model.state_dict().items()
        }

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history_rows)
history_df


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history_df['epoch'], history_df['train_loss'], label='train loss')
plt.plot(history_df['epoch'], history_df['val_loss'], label='val loss')
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Loss')
plt.title('Context-Aware Masked Reconstruction')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


## 6. Inspect Reconstructed Local Patches

In [ ]:
examples = collect_context_examples(
    model=model,
    dataloader=loaders.val,
    device=device,
    num_examples=RUN_ARGS['num_examples'],
)

examples.keys()


In [ ]:
def show_single_channel(ax, tensor, title: str):
    image = tensor.squeeze().detach().cpu().numpy()
    ax.imshow(image, cmap='gray', vmin=0.0, vmax=1.0)
    ax.set_title(title)
    ax.axis('off')

count = examples['original'].shape[0]
fig, axes = plt.subplots(count, 5, figsize=(18, 3 * count))
if count == 1:
    axes = np.expand_dims(axes, axis=0)

for row in range(count):
    show_single_channel(axes[row, 0], examples['context'][row], 'Context input')
    show_single_channel(axes[row, 1], examples['original'][row], 'Original local')
    show_single_channel(axes[row, 2], examples['masked_input'][row], 'Masked local')
    show_single_channel(axes[row, 3], examples['reconstruction'][row], 'Reconstruction')
    show_single_channel(axes[row, 4], examples['mask'][row], 'Mask')

plt.tight_layout()
plt.show()


## 7. Stitch Local Patches Back Together

This reconstructs all kept local patches for one source image and stitches them into a mosaic using the saved patch coordinates.


In [ ]:
records_by_source = group_records_by_source(records)
source_image = sorted(records_by_source)[0]
source_records = sorted(
    records_by_source[source_image],
    key=lambda record: (record.row, record.col),
)

source_dataset = HiRISEContextPatchDataset(
    source_records,
    local_input_size=RUN_ARGS['local_patch_size'],
    context_input_size=RUN_ARGS['context_input_size'],
)

stitched_original_patches, stitched_reconstructed_patches = reconstruct_dataset(
    model=model,
    dataset=source_dataset,
    device=device,
    batch_size=RUN_ARGS['batch_size'],
    use_amp=use_amp,
)

stitched_original = stitch_patch_arrays(source_records, stitched_original_patches)
stitched_reconstruction = stitch_patch_arrays(source_records, stitched_reconstructed_patches)

print('Source image:', Path(source_image).name)
print('Used patches:', len(source_records))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].imshow(stitched_original, cmap='gray', vmin=0.0, vmax=1.0)
axes[0].set_title('Stitched original local patches')
axes[0].axis('off')

axes[1].imshow(stitched_reconstruction, cmap='gray', vmin=0.0, vmax=1.0)
axes[1].set_title('Stitched reconstructed local patches')
axes[1].axis('off')

plt.tight_layout()
plt.show()
